## Objective:
#### "To analyze how student demographics, academic performance, and learning behaviors influence course completion and student success, in order to identify at-risk student profiles and recommend targeted academic support interventions."

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# ==========================================
# 1. SETUP & DATA PREP
# ==========================================
# Load Data
master_df = pd.read_csv('cleaned_data/master_dataset.csv')
course_codes = pd.read_csv('cleaned_data/course_codes_clean.csv')

# --- Merge Course Names ---
master_df['COURSE_CODE_PREFIX'] = master_df['CLASS'].astype(str).str.split('-').str[0]
master_df['COURSE_CODE_PREFIX'] = pd.to_numeric(master_df['COURSE_CODE_PREFIX'], errors='coerce')
master_df = pd.merge(master_df, course_codes, left_on='COURSE_CODE_PREFIX', right_on='CODE', how='left')

# --- Create Analytic Columns ---
# 1. Study Bins
bins = [0, 5, 10, 15, 20, 100]
labels = ['0-5 hrs', '6-10 hrs', '11-15 hrs', '16-20 hrs', '>20 hrs']
master_df['Study_Habit'] = pd.cut(master_df['SELF-STUDY HRS'], bins=bins, labels=labels)

# 2. Age Groups
master_df['Age_Group'] = pd.cut(master_df['AGE'], bins=[0, 25, 35, 45, 100], labels=['<25', '26-35', '36-45', '>45'])

# 3. Risk Status
def define_risk(row):
    if row['GPA'] < 2.0 or row['ATTENDANCE'] < 80:
        return '🔴 High Risk'
    elif row['GPA'] < 3.0:
        return '🟠 Moderate Risk'
    else:
        return '🟢 On Track'
master_df['Risk_Status'] = master_df.apply(define_risk, axis=1)

# Drop NaNs for plotting
plot_df = master_df.dropna(subset=['GPA', 'ATTENDANCE', 'SELF-STUDY HRS', 'HIGHEST QUALIFICATION']).copy()

# --- GLOBAL DARK THEME SETTING ---
pio.templates.default = "plotly_dark"


# ==========================================
# CHART 1: Demographics (Violin Plot)
# ==========================================
# Insight: Shows the shape of GPA distribution.
# Added 'box=True' to show the statistical summary inside.
fig1 = px.violin(
    plot_df,
    x='Age_Group',
    y='GPA',
    color='Age_Group',
    box=True, 
    points='all', # Shows every student as a dot
    title='<b>1. Age Analysis: GPA Distribution (Violin)</b>',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig1.show()


# ==========================================
# CHART 2: Course Difficulty (Average Bar)
# ==========================================
# Insight: CHANGED TO MEAN. Shows the true average performance.
course_perf = plot_df.groupby('COURSE NAME')[['GPA']].mean().sort_values('GPA').reset_index()

fig2 = px.bar(
    course_perf,
    y='COURSE NAME',
    x='GPA',
    orientation='h',
    title='<b>2. Course Average GPA (Difficulty Benchmark)</b>',
    text_auto='.2f',
    color='GPA',
    color_continuous_scale='Teal'
)
fig2.update_layout(xaxis_range=[0, 4.0])
fig2.show()


# ==========================================
# CHART 3: Study Habits (Box Plot Trend)
# ==========================================
# Insight: Added 'boxmean=True'. The dashed line shows the Average.
fig3 = px.box(
    plot_df,
    x='Study_Habit',
    y='GPA',
    color='Study_Habit',
    title='<b>3. The "Study Benefit": Effort vs Success</b><br><i>(Dashed line = Average GPA)</i>',
    category_orders={'Study_Habit': labels},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig3.update_traces(boxmean=True) # Shows the Mean line
fig3.show()


# ==========================================
# CHART 4: At-Risk Count (Stacked Bar)
# ==========================================
# Insight: Counts of students who are failing (Red) vs Struggling (Orange).
risk_counts = plot_df.groupby(['COURSE NAME', 'Risk_Status']).size().reset_index(name='Count')

fig4 = px.bar(
    risk_counts,
    y='COURSE NAME',
    x='Count',
    color='Risk_Status',
    orientation='h',
    title='<b>4. At-Risk Identification (Student Count)</b>',
    color_discrete_map={'🔴 High Risk': '#ff5252', '🟠 Moderate Risk': '#ffa726', '🟢 On Track': '#69f0ae'},
    barmode='stack'
)
fig4.show()


# ==========================================
# CHART 5: Hierarchy (Sunburst)
# ==========================================
# Insight: Breakdown of student profiles. Color = Average GPA of that group.
fig5 = px.sunburst(
    plot_df,
    path=['NATIONALITY_STATUS', 'HIGHEST QUALIFICATION'],
    values='GPA', # Size
    color='GPA',  # Color by Average GPA
    title='<b>5. Student Profile Hierarchy</b>',
    color_continuous_scale='Blues'
)
fig5.show()


# ==========================================
# CHART 6: Drivers (Heatmap)
# ==========================================
# Insight: Correlation Matrix (Mathematical proof of what matters).
corr_cols = ['GPA', 'ATTENDANCE', 'SELF-STUDY HRS', 'AGE', 'PRIOR KNOWLEDGE']
corr_matrix = plot_df[corr_cols].corr()

fig6 = px.imshow(
    corr_matrix,
    text_auto='.2f',
    title='<b>6. Success Drivers (Correlation Matrix)</b>',
    color_continuous_scale='RdBu_r', # Red = Positive Correlation
    aspect='auto'
)
fig6.show()


# ==========================================
# CHART 7: Progression Trend (Line Chart)
# ==========================================
# Insight: CHANGED TO MEAN. Shows how the class average moves over time.
progression = plot_df.groupby('PERIOD')[['GPA']].mean().reset_index()
sem_order = ["Sem 1", "Sem 2", "Sem 3", "Sem 4"]

fig7 = px.line(
    progression,
    x='PERIOD',
    y='GPA',
    markers=True,
    title='<b>7. Cohort Progression: Average GPA over Time</b>',
    category_orders={"PERIOD": sem_order}
)
fig7.update_traces(line_color='#00e676', line_width=4)
fig7.show()


# ==========================================
# CHART 8: Survey Sentiment (Grouped Bar)
# ==========================================
# Insight: CHANGED TO MEAN. Average sentiment score (1-5).
survey_cols = ['PRIOR KNOWLEDGE', 'TEACHING SUPPORT', 'COURSE RELEVANCE']
survey_data = plot_df.groupby('COURSE NAME')[survey_cols].mean().reset_index()
survey_melted = survey_data.melt(id_vars='COURSE NAME', var_name='Metric', value_name='Avg Score')

fig8 = px.bar(
    survey_melted,
    x='COURSE NAME',
    y='Avg Score',
    color='Metric',
    barmode='group',
    title='<b>8. Student Feedback: Average Survey Scores</b>',
    text_auto='.1f',
    color_discrete_sequence=['#448aff', '#b388ff', '#ff80ab']
)
fig8.show()


# ==========================================
# REPLACEMENT CHART 9: Attendance Impact (Bar Chart)
# ==========================================
# Insight: "Downey-proof" logic. Group attendance into 3 simple buckets.
# It shows a clear step down: Low Attendance = Low Grades.
bins_att = [0, 80, 90, 100]
labels_att = ['<80% (Critical)', '80-90% (Warning)', '90-100% (Good)']
plot_df['Attendance_Group'] = pd.cut(plot_df['ATTENDANCE'], bins=bins_att, labels=labels_att)

att_perf = plot_df.groupby('Attendance_Group')[['GPA']].mean().reset_index()

fig9 = px.bar(
    att_perf,
    x='Attendance_Group',
    y='GPA',
    title='<b>9. The Attendance Effect: Low Attendance = Low GPA</b>',
    text_auto='.2f',
    color='GPA',
    color_continuous_scale='Redor' # Red to Orange
)
fig9.update_layout(yaxis_range=[1.0, 4.0]) # Zoom in to make differences obvious
fig9.show()


# ==========================================
# NEW CHART 10: Nationality Performance Gap
# ==========================================
# Insight: Do Foreigners actually do better? (Simple Comparison)
nat_perf = plot_df.groupby('NATIONALITY_STATUS')[['GPA']].mean().sort_values('GPA', ascending=False).reset_index()

fig10 = px.bar(
    nat_perf,
    x='NATIONALITY_STATUS',
    y='GPA',
    title='<b>10. Demographic Analysis: Local vs Foreigner Gap</b>',
    text_auto='.2f',
    color='NATIONALITY_STATUS',
    color_discrete_sequence=['#00e676', '#2979ff', '#ff9100'] # Distinct colors
)
fig10.update_layout(yaxis_range=[2.5, 4.0])
fig10.show()


# ==========================================
# NEW CHART 11: Prior Knowledge Benefit
# ==========================================
# Insight: Does knowing the topic beforehand help? (Trend)
# We group the 1-5 rating and see if GPA goes up.
prior_perf = plot_df.groupby('PRIOR KNOWLEDGE')[['GPA']].mean().reset_index()

fig11 = px.bar(
    prior_perf,
    x='PRIOR KNOWLEDGE',
    y='GPA',
    title='<b>11. The Head Start: Does Prior Knowledge Boost GPA?</b>',
    text_auto='.2f',
    labels={'PRIOR KNOWLEDGE': 'Student Self-Rating (1-5)'},
    color='GPA',
    color_continuous_scale='Bluyl' # Blue to Yellow
)
fig11.update_layout(yaxis_range=[2.5, 4.0])
fig11.show()


# ==========================================
# NEW CHART 12: Support System Analysis
# ==========================================
# Insight: Who helps more? Teachers or Company? 
# This helps with the "Recommendation" part of your objective.
support_cols = ['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT']
support_data = plot_df[support_cols].mean().reset_index()
support_data.columns = ['Support Type', 'Average Score']

fig12 = px.bar(
    support_data.sort_values('Average Score'),
    x='Average Score',
    y='Support Type',
    orientation='h',
    title='<b>12. Support Systems: Where do students feel supported?</b>',
    text_auto='.2f',
    color='Average Score',
    color_continuous_scale='Purples'
)
fig12.show()

C:\Users\User\AppData\Local\Temp\ipykernel_34184\2446839073.py:198: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [2]:
# ==========================================
# COMBINED CHART 1: The "Effort Gap"
# ==========================================
# Insight: "Myth Busted/Confirmed: Do Foreigners just work harder?"
fig_combo1 = px.box(
    plot_df, 
    x='NATIONALITY_STATUS', 
    y='SELF-STUDY HRS', 
    color='NATIONALITY_STATUS',
    title='<b>Combined Insight 1: Do Foreigners Study More?</b><br><i>(Comparing Effort Levels by Nationality)</i>',
    points="all", # Show the dots to be transparent
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig_combo1.show()


# ==========================================
# COMBINED CHART 2: The "Demographic Risk"
# ==========================================
# Insight: "Visualizing the Age Cliff"
# We count risk status within each age group
age_risk = plot_df.groupby(['Age_Group', 'Risk_Status']).size().reset_index(name='Count')

# Calculate percentages for a fair comparison (100% Stacked Bar)
age_totals = age_risk.groupby('Age_Group')['Count'].transform('sum')
age_risk['Percentage'] = age_risk['Count'] / age_totals * 100

fig_combo2 = px.bar(
    age_risk, 
    x='Age_Group', 
    y='Percentage', 
    color='Risk_Status', 
    title='<b>Combined Insight 2: Risk Profile by Age (Normalized)</b><br><i>(Notice how the Red Bar grows with Age)</i>',
    color_discrete_map={'🔴 High Risk': '#ff5252', '🟠 Moderate Risk': '#ffa726', '🟢 On Track': '#69f0ae'},
    text_auto='.1f'
)
fig_combo2.show()


# ==========================================
# COMBINED CHART 3: The "Course Quadrant" (360 View)
# ==========================================
# Insight: "The Problem Child Analysis"
# We aggregate EVERYTHING by Course
course_stats = plot_df.groupby('COURSE NAME').agg({
    'GPA': 'mean',
    'ATTENDANCE': 'mean',
    'Risk_Status': lambda x: (x == '🔴 High Risk').sum(), # Count high risk students
    'STUDENT ID': 'count' # Total students
}).reset_index()
course_stats.rename(columns={'Risk_Status': 'High_Risk_Count', 'STUDENT ID': 'Total_Students'}, inplace=True)

fig_combo3 = px.scatter(
    course_stats, 
    x='ATTENDANCE', 
    y='GPA', 
    size='High_Risk_Count', # Big bubbles = More failing students
    color='COURSE NAME',
    text='COURSE NAME', # Label the bubbles
    title='<b>Combined Insight 3: Course Performance Matrix</b><br><i>(Size = Number of High Risk Students)</i>',
)
# Add quadrants
avg_gpa = plot_df['GPA'].mean()
avg_att = plot_df['ATTENDANCE'].mean()
fig_combo3.add_hline(y=avg_gpa, line_dash="dash", line_color="white", annotation_text="Avg GPA")
fig_combo3.add_vline(x=avg_att, line_dash="dash", line_color="white", annotation_text="Avg Att")
fig_combo3.update_traces(textposition='top center')
fig_combo3.show()

C:\Users\User\AppData\Local\Temp\ipykernel_34184\1848090297.py:22: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_34184\1848090297.py:25: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [3]:
# ==========================================
# STRONG COMBO 1: The "Compensation Effect"
# ==========================================
# Insight: "Can hard work (Self-Study) save you if you skip class?"
# We bin Attendance and Study Hours to see the interaction.
# Result: Shows if high study hours can fix low attendance.
print("Generating Chart 1...")
fig_combo4 = px.density_heatmap(
    plot_df, 
    x='ATTENDANCE', 
    y='SELF-STUDY HRS', 
    z='GPA', 
    histfunc='avg', # Colors the block by Average GPA
    nbinsx=10,      # Fewer bins = Bigger blocks (easier to see)
    nbinsy=10,
    title='<b>1. The Compensation Matrix: Impact of Attendance vs Study on GPA</b><br><i>(Red/Orange = Failing Zone, Green = Passing Zone)</i>',
    labels={'ATTENDANCE': 'Attendance %', 'SELF-STUDY HRS': 'Study Hours/Week', 'GPA': 'Avg GPA'},
    color_continuous_scale='RdYlGn', # Red to Green
    text_auto='.1f' # Show GPA numbers in the blocks
)
fig_combo4.update_layout(xaxis_range=[50, 100], yaxis_range=[0, 30])
fig_combo4.show()


# ==========================================
# STRONG COMBO 2: The "Demographic Reality Check"
# ==========================================
# Insight: "Are Foreigners smarter, or just more qualified?"
# We compare Locals vs Foreigners, but SPLIT by Qualification.
# If Foreigners (Blue box) are higher than Locals (Red box) AT THE SAME LEVEL, 
# then it is a true work ethic difference, not just background.
fig_combo5 = px.box(
    plot_df, 
    x='NATIONALITY_STATUS', 
    y='GPA', 
    color='HIGHEST QUALIFICATION',
    title='<b>Strong Insight 2: The Reality Check (Nationality vs Qualification)</b><br><i>(Is the "Foreigner Advantage" real across all education levels?)</i>',
    category_orders={"HIGHEST QUALIFICATION": ["Certificate", "Diploma", "Degree", "Masters"]},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_combo5.show()


# ==========================================
# STRONG COMBO 3: The "Trajectory of Failure"
# ==========================================
# Insight: "Do At-Risk students recover over time?"
# We track the GPA of the 3 Risk Groups across semesters.
# If the Red Line goes DOWN, your intervention needs to happen in Sem 1.
# If the Red Line goes UP, the current support is working.
risk_trend = plot_df.groupby(['PERIOD', 'Risk_Status'])['GPA'].mean().reset_index()
sem_order = ["Sem 1", "Sem 2", "Sem 3", "Sem 4"]

fig_combo6 = px.line(
    risk_trend, 
    x='PERIOD', 
    y='GPA', 
    color='Risk_Status',
    markers=True,
    title='<b>Strong Insight 3: The Trajectory (Do Failing Students Recover?)</b><br><i>(Performance path of High Risk vs On Track students)</i>',
    category_orders={"PERIOD": sem_order},
    color_discrete_map={'🔴 High Risk': '#ff5252', '🟠 Moderate Risk': '#ffa726', '🟢 On Track': '#69f0ae'}
)
fig_combo6.update_traces(line_width=5)
fig_combo6.show()

Generating Chart 1...


In [4]:
# ==========================================
# ROOT CAUSE 1: The "Absenteeism Profile"
# ==========================================
# Question: WHO is skipping class? 
# Insight: If older students skip more, your intervention is "Flexible Schedules."
# If younger students skip more, your intervention is "Stricter Discipline."
fig_cause1 = px.box(
    plot_df, 
    x='Age_Group', 
    y='ATTENDANCE', 
    color='Age_Group',
    title='<b>Root Cause 1: Who is skipping class? (Attendance by Age)</b><br><i>(Identify the demographic with the lowest discipline)</i>',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_cause1.update_layout(yaxis_range=[50, 100]) # Zoom in on the problem area
fig_cause1.show()


# ==========================================
# ROOT CAUSE 2: The "Support Impact"
# ==========================================
# Question: Does Company Support actually motivate them to study?
# Insight: If this graph goes UP, it proves that "Employer Buy-in" is key to student success.
# We group Company Support (1-5) and check Self-Study Hours.
support_study = plot_df.groupby('COMPANY SUPPORT')['SELF-STUDY HRS'].median().reset_index()

fig_cause2 = px.bar(
    support_study, 
    x='COMPANY SUPPORT', 
    y='SELF-STUDY HRS',
    title='<b>Root Cause 2: Does Company Support drive Motivation?</b><br><i>(Median Self-Study Hours vs Company Support Rating)</i>',
    labels={'COMPANY SUPPORT': 'Company Support Rating (1-5)', 'SELF-STUDY HRS': 'Median Study Hours'},
    color='SELF-STUDY HRS',
    color_continuous_scale='Blues'
)
fig_cause2.show()


# ==========================================
# ROOT CAUSE 3: The "Curriculum Gap"
# ==========================================
# Question: Are we teaching them things they already know?
# Insight: Compare "Prior Knowledge" vs "Course Relevance".
# If they have High Prior Knowledge but Low Relevance, the course is redundant (waste of time).
# This is a "Quality Assurance" insight for your lecturer.
fig_cause3 = px.density_heatmap(
    plot_df, 
    x='PRIOR KNOWLEDGE', 
    y='COURSE RELEVANCE', 
    title='<b>Root Cause 3: Curriculum Quality Check</b><br><i>(Do students with high knowledge find the course relevant?)</i>',
    labels={'PRIOR KNOWLEDGE': 'Prior Knowledge (1-5)', 'COURSE RELEVANCE': 'Course Relevance (1-5)'},
    color_continuous_scale='Purples',
    text_auto=True
)
fig_cause3.show()

In [44]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_dark"

# 1. LOAD & CLEAN DATA
try:
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
except FileNotFoundError:
    master_df = pd.read_csv('master_dataset.csv')

plot_df = master_df.copy()
plot_df['GPA'] = pd.to_numeric(plot_df['GPA'], errors='coerce')

# ==========================================
# GRAPH B: IMPACT GAP ANALYSIS
# ==========================================
rating_cols = ["PRIOR KNOWLEDGE", "COURSE RELEVANCE", "TEACHING SUPPORT", "COMPANY SUPPORT", "FAMILY SUPPORT"]
impact_data = []

for col in rating_cols:
    plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
    
    # Calculate Averages
    low_gpa = plot_df[plot_df[col].isin([1, 2])]['GPA'].mean()
    mid_gpa = plot_df[plot_df[col] == 3]['GPA'].mean()
    high_gpa = plot_df[plot_df[col].isin([4, 5])]['GPA'].mean()
    gap = high_gpa - low_gpa
    
    impact_data.append({
        'Factor': col.title(),
        'Low Support GPA': low_gpa,
        'Mid Support GPA': mid_gpa,
        'High Support GPA': high_gpa,
        'Gap': gap
    })

# Sort by Gap
impact_df = pd.DataFrame(impact_data).sort_values('Gap', ascending=True)

fig_gap = go.Figure()

# 1. Draw Lines
for i, row in impact_df.iterrows():
    fig_gap.add_trace(go.Scatter(
        x=[row['Low Support GPA'], row['High Support GPA']],
        y=[row['Factor'], row['Factor']],
        mode='lines',
        line=dict(color='grey', width=3),
        showlegend=False,
        hoverinfo='skip'
    ))

# 2. Draw Red Dots (Low Support)
fig_gap.add_trace(go.Scatter(
    x=impact_df['Low Support GPA'],
    y=impact_df['Factor'],
    mode='markers+text',
    name='Low Support (1-2)',
    marker=dict(color='#ff5252', size=14, line=dict(width=2, color='white')),
    texttemplate='%{x:.2f}',
    textposition='middle left',
    textfont=dict(color='#ff5252'),
    hovertemplate="<b>%{y}</b><br>Low: %{x:.2f}<extra></extra>"
))

# 3. Draw Yellow Dots (Mid Support)
fig_gap.add_trace(go.Scatter(
    x=impact_df['Mid Support GPA'],
    y=impact_df['Factor'],
    mode='markers+text',
    name='Mid Support (3)',
    marker=dict(color='#ffab00', size=14, line=dict(width=2, color='white'), symbol='diamond'),
    texttemplate='%{x:.2f}',
    textposition='top center',
    textfont=dict(color='#ffab00'),
    hovertemplate="<b>%{y}</b><br>Mid: %{x:.2f}<extra></extra>"
))

# 4. Draw Green Dots (High Support)
fig_gap.add_trace(go.Scatter(
    x=impact_df['High Support GPA'],
    y=impact_df['Factor'],
    mode='markers+text',
    name='High Support (4-5)',
    marker=dict(color='#00e676', size=14, line=dict(width=2, color='white')),
    texttemplate='%{x:.2f}',
    textposition='middle right',
    textfont=dict(color='#00e676'),
    hovertemplate="<b>%{y}</b><br>High: %{x:.2f}<extra></extra>"
))

# 5. Layout Polish
fig_gap.update_layout(
    title='<b>Identifying Which Support Factors Drive the Biggest Improvement in GPA</b><br><i>(We compare the performance gap between students with low vs. high support)</i>',
    xaxis=dict(
        title="Average GPA", 
        range=[1.8, 4.2],
        showticklabels=True
    ),
    yaxis=dict(title=""),
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
    height=600,
    margin=dict(l=150, b=100, r=50)
)

fig_gap.show()

In [48]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_dark"

# 1. LOAD DATA
try:
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
except FileNotFoundError:
    master_df = pd.read_csv('master_dataset.csv')

plot_df = master_df.copy()
plot_df['GPA'] = pd.to_numeric(plot_df['GPA'], errors='coerce')

# ==========================================
# GRAPH B: EFFORT vs ENVIRONMENT (Smart Labelling)
# ==========================================
impact_data = []

# --- PART 1: Support Factors (Likert 1-5) ---
rating_cols = ["PRIOR KNOWLEDGE", "COURSE RELEVANCE", "TEACHING SUPPORT", "COMPANY SUPPORT", "FAMILY SUPPORT"]
for col in rating_cols:
    plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
    low_gpa = plot_df[plot_df[col].isin([1, 2])]['GPA'].mean()
    mid_gpa = plot_df[plot_df[col] == 3]['GPA'].mean()
    high_gpa = plot_df[plot_df[col].isin([4, 5])]['GPA'].mean()
    gap = high_gpa - low_gpa
    impact_data.append({'Factor': col.title(), 'Low': low_gpa, 'Mid': mid_gpa, 'High': high_gpa, 'Gap': gap})

# --- PART 2: Study Effort (Hours) ---
s = pd.to_numeric(plot_df['SELF-STUDY HRS'], errors='coerce')
low_study_gpa = plot_df[s <= 10]['GPA'].mean()
mid_study_gpa = plot_df[(s >= 11) & (s <= 17)]['GPA'].mean()
high_study_gpa = plot_df[s >= 18]['GPA'].mean()
study_gap = high_study_gpa - low_study_gpa

impact_data.append({
    'Factor': '<b>Self-Study Effort</b>',
    'Low': low_study_gpa, 'Mid': mid_study_gpa, 'High': high_study_gpa, 'Gap': study_gap
})

# --- PART 3: CREATE CUSTOM LABELS ---
impact_df = pd.DataFrame(impact_data).sort_values('Gap', ascending=True)

# Create text columns so we can customize the Study Effort row specifically
impact_df['Low_Text'] = impact_df['Low'].apply(lambda x: f"{x:.2f}")
impact_df['Mid_Text'] = impact_df['Mid'].apply(lambda x: f"{x:.2f}")
impact_df['High_Text'] = impact_df['High'].apply(lambda x: f"{x:.2f}")

# Target ONLY the "Self-Study" row and append the hours
mask = impact_df['Factor'] == '<b>Self-Study Effort</b>'
impact_df.loc[mask, 'Low_Text'] += "<br>(5-10h)"
impact_df.loc[mask, 'Mid_Text'] += "<br>(11-17h)"
impact_df.loc[mask, 'High_Text'] += "<br>(18+h)"

# --- PART 4: PLOT ---
fig_gap = go.Figure()

# Draw Lines
for i, row in impact_df.iterrows():
    fig_gap.add_trace(go.Scatter(
        x=[row['Low'], row['High']], y=[row['Factor'], row['Factor']],
        mode='lines', line=dict(color='grey', width=3), showlegend=False, hoverinfo='skip'
    ))

# Draw Dots (Using 'text' argument for custom labels)
fig_gap.add_trace(go.Scatter(x=impact_df['Low'], y=impact_df['Factor'], mode='markers+text', 
    name='Low Support (1-2)', # Legend stays Clean
    marker=dict(color='#ff5252', size=14, line=dict(width=2, color='white')),
    text=impact_df['Low_Text'], texttemplate='%{text}', textposition='middle left', textfont=dict(color='#ff5252')))

fig_gap.add_trace(go.Scatter(x=impact_df['Mid'], y=impact_df['Factor'], mode='markers+text', 
    name='Mid Support (3)', 
    marker=dict(color='#ffab00', size=14, line=dict(width=2, color='white'), symbol='diamond'),
    text=impact_df['Mid_Text'], texttemplate='%{text}', textposition='top center', textfont=dict(color='#ffab00')))

fig_gap.add_trace(go.Scatter(x=impact_df['High'], y=impact_df['Factor'], mode='markers+text', 
    name='High Support (4-5)', 
    marker=dict(color='#00e676', size=14, line=dict(width=2, color='white')),
    text=impact_df['High_Text'], texttemplate='%{text}', textposition='middle right', textfont=dict(color='#00e676')))

# Fail Zone
fig_gap.add_vrect(x0=0, x1=2.0, fillcolor="red", opacity=0.15, layer="below", line_width=0, annotation_text="FAIL ZONE", annotation_font=dict(color="red"))

# Layout
fig_gap.update_layout(
    title='<b>Effort vs. Environment: Drivers of Student Success</b><br><i>(We compare the impact of Support Systems vs. Self-Study Hours)</i>',
    xaxis=dict(title="Average GPA", range=[1.5, 4.1], showticklabels=True),
    yaxis=dict(title=""),
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
    height=600, margin=dict(l=150, b=100, r=60) # Increased right margin for text
)
fig_gap.show()

# Graphs that maybe can be used for dashboards

In [46]:
# Check the stats
print("--- STATS ---")
print(master_df['SELF-STUDY HRS'].describe())

# Check the exact unique values (to see if they are 1, 2, 3... or 10, 20...)
print("\n--- UNIQUE VALUES ---")
print(sorted(master_df['SELF-STUDY HRS'].unique()))

# See the distribution (How many students in each hour bucket?)
print("\n--- DISTRIBUTION ---")
print(master_df['SELF-STUDY HRS'].value_counts().sort_index())

--- STATS ---
count    494.000000
mean      13.338057
std        4.225525
min        5.000000
25%       10.000000
50%       13.000000
75%       17.000000
max       23.000000
Name: SELF-STUDY HRS, dtype: float64

--- UNIQUE VALUES ---
[np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(nan), np.float64(21.0), np.float64(22.0), np.float64(23.0)]

--- DISTRIBUTION ---
SELF-STUDY HRS
5.0     16
6.0     17
7.0     23
8.0     18
9.0     22
10.0    34
11.0    33
12.0    59
13.0    29
14.0    33
15.0    47
16.0    35
17.0    30
18.0    33
19.0    28
20.0    31
21.0     1
22.0     3
23.0     2
Name: count, dtype: int64


In [54]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_dark"

# 1. LOAD & CLEAN DATA
try:
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
except FileNotFoundError:
    master_df = pd.read_csv('master_dataset.csv')

plot_df = master_df.copy()
plot_df['GPA'] = pd.to_numeric(plot_df['GPA'], errors='coerce')

# ==============================================================================
# GRAPH 1: IMPACT GAP (Support Factors Only)
# ==============================================================================
rating_cols = ["PRIOR KNOWLEDGE", "COURSE RELEVANCE", "TEACHING SUPPORT", "COMPANY SUPPORT", "FAMILY SUPPORT"]
impact_data = []

for col in rating_cols:
    plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
    low_gpa = plot_df[plot_df[col].isin([1, 2])]['GPA'].mean()
    mid_gpa = plot_df[plot_df[col] == 3]['GPA'].mean()
    high_gpa = plot_df[plot_df[col].isin([4, 5])]['GPA'].mean()
    gap = high_gpa - low_gpa
    impact_data.append({'Factor': col.title(), 'Low': low_gpa, 'Mid': mid_gpa, 'High': high_gpa, 'Gap': gap})

impact_df = pd.DataFrame(impact_data).sort_values('Gap', ascending=True)

fig1 = go.Figure()

# Lines
for i, row in impact_df.iterrows():
    fig1.add_trace(go.Scatter(x=[row['Low'], row['High']], y=[row['Factor'], row['Factor']], mode='lines', line=dict(color='grey', width=3), showlegend=False, hoverinfo='skip'))

# Dots
fig1.add_trace(go.Scatter(x=impact_df['Low'], y=impact_df['Factor'], mode='markers+text', name='Low Support (1-2)', 
    marker=dict(color='#ff5252', size=14, line=dict(width=2, color='white')), texttemplate='%{x:.2f}', textposition='middle left', textfont=dict(color='#ff5252')))
fig1.add_trace(go.Scatter(x=impact_df['Mid'], y=impact_df['Factor'], mode='markers+text', name='Mid Support (3)', 
    marker=dict(color='#ffab00', size=14, line=dict(width=2, color='white'), symbol='diamond'), texttemplate='%{x:.2f}', textposition='top center', textfont=dict(color='#ffab00')))
fig1.add_trace(go.Scatter(x=impact_df['High'], y=impact_df['Factor'], mode='markers+text', name='High Support (4-5)', 
    marker=dict(color='#00e676', size=14, line=dict(width=2, color='white')), texttemplate='%{x:.2f}', textposition='middle right', textfont=dict(color='#00e676')))

fig1.update_layout(
    title='<b>1. Environment Impact: Which Support Factor Matters Most?</b><br><i>(Gap between Low and High Support)</i>',
    xaxis=dict(title="Average GPA", range=[2.0, 4.0]), yaxis=dict(title=""),
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"), height=500, margin=dict(l=150, b=100)
)
fig1.show()


# ==============================================================================
# GRAPH 2: SELF-STUDY HOURS (Line Chart)
# ==============================================================================
# Calculate Mean GPA per Hour
plot_df['SELF-STUDY HRS'] = pd.to_numeric(plot_df['SELF-STUDY HRS'], errors='coerce')
study_trend = plot_df.groupby('SELF-STUDY HRS')['GPA'].mean().reset_index()
study_trend = study_trend[study_trend['SELF-STUDY HRS'] <= 23] # Filter max range

fig2 = go.Figure()

# The Trend Line
fig2.add_trace(go.Scatter(
    x=study_trend['SELF-STUDY HRS'], 
    y=study_trend['GPA'],
    mode='lines+markers',
    name='Average GPA',
    line=dict(color='#2979ff', width=4), # Bright Blue Line
    marker=dict(size=10, color='white', line=dict(width=2, color='#2979ff')),
    hovertemplate="<b>%{x:.0f} Hours</b><br>Avg GPA: %{y:.2f}<extra></extra>"
))

fig2.update_layout(
    title='<b>2. The Effort Curve: How Self-Study Drives GPA</b><br><i>(Notice the trend: Does GPA keep rising as hours increase?)</i>',
    xaxis=dict(title="Weekly Self-Study Hours", tickmode='linear', dtick=2),
    yaxis=dict(title="Average GPA", range=[2.0, 4.2]),
    height=500,
    showlegend=False
)

fig2.show()

In [56]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

# 1. LOAD & CLEAN DATA
try:
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
    course_codes = pd.read_csv('cleaned_data/course_codes_clean.csv')
except FileNotFoundError:
    master_df = pd.read_csv('master_dataset.csv')
    course_codes = pd.read_csv('course_codes_clean.csv')

# Prep: Merge Course Names
master_df['COURSE_CODE_PREFIX'] = master_df['CLASS'].astype(str).str.split('-').str[0]
master_df['COURSE_CODE_PREFIX'] = pd.to_numeric(master_df['COURSE_CODE_PREFIX'], errors='coerce')
master_df = pd.merge(master_df, course_codes, left_on='COURSE_CODE_PREFIX', right_on='CODE', how='left')

# Prep: Define Risk Categories
def define_risk(row):
    if row['GPA'] < 2.0: return '🔴 Failing (<2.0)'
    elif row['GPA'] < 3.0: return '🟠 At Risk (2.0-3.0)'
    else: return '🟢 On Track (>3.0)'

master_df['Risk_Status'] = master_df.apply(define_risk, axis=1)

# Prep: Age Groups
master_df['Age_Group'] = pd.cut(master_df['AGE'], bins=[0, 25, 35, 45, 100], labels=['<25', '26-35', '36-45', '>45'])
plot_df = master_df.dropna(subset=['GPA', 'COURSE NAME']).copy()

# ==============================================================================
# GRAPH 3: THE DIAGNOSIS (Failure Rate by Course)
# ==============================================================================
# 1. Group raw data to get counts
risk_counts = plot_df.groupby(['COURSE NAME', 'Risk_Status']).size().reset_index(name='Count')

# 2. Calculate totals using the SUMMARY table (risk_counts), NOT the raw table (plot_df)
total_counts = risk_counts.groupby('COURSE NAME')['Count'].transform('sum')

# 3. Calc Percentage
risk_counts['Percentage'] = (risk_counts['Count'] / total_counts * 100).round(1)

fig3 = px.bar(
    risk_counts,
    y='COURSE NAME',
    x='Percentage',
    color='Risk_Status',
    orientation='h',
    title='<b>3. The Diagnosis: Which Course has the Highest Failure Rate?</b><br><i>(Red bars indicate the courses needing immediate intervention)</i>',
    text_auto=True,
    color_discrete_map={
        '🔴 Failing (<2.0)': '#ff5252', 
        '🟠 At Risk (2.0-3.0)': '#ffab00', 
        '🟢 On Track (>3.0)': '#00e676'
    }
)

fig3.update_layout(
    xaxis_title="Percentage of Students (%)", 
    xaxis_range=[0, 100], 
    yaxis_title="",
    height=450,
    legend=dict(orientation="h", y=-0.2, title="")
)
fig3.show()


# ==============================================================================
# GRAPH 4: THE PROFILE (Risk by Age Group)
# ==============================================================================
age_risk = plot_df.groupby(['Age_Group', 'Risk_Status']).size().reset_index(name='Count')

# FIX: Same fix here (Use age_risk, not plot_df)
total_age = age_risk.groupby('Age_Group')['Count'].transform('sum')

age_risk['Percentage'] = (age_risk['Count'] / total_age * 100).round(1)

fig4 = px.bar(
    age_risk,
    x='Age_Group',
    y='Percentage',
    color='Risk_Status',
    title='<b>4. The Profile: Who is Most At-Risk? (The Age Cliff)</b><br><i>(Notice how the Red/Orange bars grow larger as students get older)</i>',
    text_auto=True,
    barmode='group', 
    color_discrete_map={
        '🔴 Failing (<2.0)': '#ff5252', 
        '🟠 At Risk (2.0-3.0)': '#ffab00', 
        '🟢 On Track (>3.0)': '#00e676'
    }
)

fig4.update_layout(
    xaxis_title="Age Group",
    yaxis_title="Percentage of Students (%)", 
    height=450,
    legend=dict(orientation="h", y=-0.2, title="")
)
fig4.show()

C:\Users\User\AppData\Local\Temp\ipykernel_34184\2732188514.py:72: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_34184\2732188514.py:75: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [68]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

# 1. LOAD & CLEAN DATA
try:
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
    course_codes = pd.read_csv('cleaned_data/course_codes_clean.csv')
except FileNotFoundError:
    master_df = pd.read_csv('master_dataset.csv')
    course_codes = pd.read_csv('course_codes_clean.csv')

# Prep
master_df['COURSE_CODE_PREFIX'] = master_df['CLASS'].astype(str).str.split('-').str[0]
master_df['COURSE_CODE_PREFIX'] = pd.to_numeric(master_df['COURSE_CODE_PREFIX'], errors='coerce')
master_df = pd.merge(master_df, course_codes, left_on='COURSE_CODE_PREFIX', right_on='CODE', how='left')
plot_df = master_df.dropna(subset=['GPA', 'COURSE NAME']).copy()

# 2. CREATE ADULT LEARNER GROUPS (Career Stage)
plot_df['Age_Group'] = pd.cut(
    plot_df['AGE'], 
    bins=[0, 35, 45, 100], 
    labels=['Early Career (<35)', 'Mid-Career (35-45)', 'Senior (>45)']
)

# ==============================================================================
# GRAPH 3: PERFORMANCE DIAGNOSTICS (With Dual Thresholds)
# ==============================================================================

# 1. Identify Bottom 5 Courses
bottom_5_courses = plot_df.groupby('COURSE NAME')['GPA'].mean().nsmallest(5).index.tolist()
focused_df = plot_df[plot_df['COURSE NAME'].isin(bottom_5_courses)]

# 2. Calculate Average GPA
perf_data = focused_df.groupby(['COURSE NAME', 'Age_Group'], observed=False)['GPA'].mean().reset_index()

# 3. Plot
fig_combined = px.bar(
    perf_data,
    x='COURSE NAME',
    y='GPA',
    color='Age_Group',
    barmode='group', 
    title='<b>Performance Gap: GPA by Career Stage</b><br><i>(Comparing Early vs. Senior professionals in the lowest-performing courses)</i>',
    text_auto='.2f', 
    labels={'GPA': 'Average GPA', 'COURSE NAME': ''},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# --- LAYOUT FIXES ---
fig_combined.update_layout(
    yaxis_range=[0, 4.0], # Hard limit at Max GPA
    height=550,           # Slightly taller
    
    # LEGEND ADJUSTMENT
    legend=dict(
        orientation="h", 
        y=-0.4,          # Push legend WAY down
        x=0.5, 
        xanchor="center",
        title="Career Stage"
    ),
    
    # MARGIN ADJUSTMENT
    margin=dict(b=150),  # Add extra empty space at the bottom for the legend
    
    xaxis=dict(tickangle=-15) # Tilted labels for readability
)

fig_combined.show()

In [64]:
# 1. Check Basic Stats
print("--- AGE STATISTICS ---")
print(master_df['AGE'].describe())

# 2. Check the Actual Unique Ages
print("\n--- UNIQUE AGES (Sorted) ---")
print(sorted(master_df['AGE'].unique()))

# 3. Check the Counts per Age Group
print("\n--- COUNT PER AGE GROUP ---")
print(master_df['Age_Group'].value_counts().sort_index())

--- AGE STATISTICS ---
count    516.000000
mean      41.275388
std        9.029313
min       15.300000
25%       34.075000
50%       41.200000
75%       48.400000
max       64.800000
Name: AGE, dtype: float64

--- UNIQUE AGES (Sorted) ---
[np.float64(15.3), np.float64(24.1), np.float64(24.8), np.float64(25.6), np.float64(25.7), np.float64(25.8), np.float64(25.9), np.float64(26.3), np.float64(26.6), np.float64(26.7), np.float64(26.8), np.float64(27.0), np.float64(27.3), np.float64(27.6), np.float64(27.7), np.float64(27.8), np.float64(28.3), np.float64(28.4), np.float64(28.5), np.float64(29.1), np.float64(29.4), np.float64(29.9), np.float64(30.5), np.float64(31.1), np.float64(31.2), np.float64(31.4), np.float64(31.8), np.float64(32.0), np.float64(32.1), np.float64(32.2), np.float64(32.3), np.float64(33.1), np.float64(33.3), np.float64(33.5), np.float64(33.8), np.float64(34.0), np.float64(34.1), np.float64(34.4), np.float64(34.9), np.float64(35.0), np.float64(35.2), np.float64(35.3), np.f

In [72]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

# 1. LOAD & CLEAN
try:
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
except FileNotFoundError:
    master_df = pd.read_csv('master_dataset.csv')

plot_df = master_df.dropna(subset=['GPA', 'ATTENDANCE']).copy()

# ==============================================================================
# GRAPH 5: THE GENERIC STANDARD (Attendance Ranges vs GPA)
# ==============================================================================

# 1. Create Simple Bins (The "Generic" Groups)
# We use standard cutoffs: Below 80 (Fail Risk), 80-90 (Good), 90-100 (Excellent)
bins = [0, 80, 90, 101]
labels = ['Below 80% (Low)', '80% - 90% (Avg)', '90% - 100% (High)']
plot_df['Attendance_Range'] = pd.cut(plot_df['ATTENDANCE'], bins=bins, labels=labels, right=False)

# 2. Calculate Average GPA per Range
att_trend = plot_df.groupby('Attendance_Range', observed=False)['GPA'].mean().reset_index()

# 3. Plot Simple Bar Chart
fig_generic = px.bar(
    att_trend,
    x='Attendance_Range',
    y='GPA',
    title='<b>5. Attendance Impact: Does showing up matter?</b><br><i>(Average GPA by Attendance Level)</i>',
    text_auto='.2f', # Show the exact GPA number
    color='GPA',     # Color gradient makes the "Low" bar look obviously different
    color_continuous_scale='RdYlGn' # Red (Low) to Green (High)
)

fig_generic.update_layout(
    yaxis=dict(range=[0, 4.0], title="Average GPA"),
    xaxis=dict(title="Attendance Rate"),
    height=500,
    showlegend=False,
    coloraxis_showscale=False # Hide color bar to keep it clean
)

fig_generic.show()